<!-- GENERATED by scripts/build_notebooks.py from _variables.yml. Do not edit this cell. -->
# 0 · Setup and hello RL

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/practical-rl/blob/main/notebooks/00-setup.ipynb)

**Practical Reinforcement Learning** · Pre-work (Module 0) · core 20 min · designed for Google Colab, CPU runtime

Check your run path, take the entry check, and run a random agent on CartPole with seed bands.

**By the end of this notebook you can:**

- Run the setup notebook on at least one run path and read its versions and runtime.
- Plot returns of a random CartPole agent with seed bands.
- Produce and submit a run record.

**How the lab works.** Each exercise: *Predict* (write your guess first) → a `# TODO` cell you complete → *Run* → *Explain* → a checkpoint that passes or fails. A folded Hint and Solution sit under each TODO. The solution never replaces your code; if you are stuck, run `lab.use_reference(N)` to continue with it.

Briefing: [https://project-delphi.github.io/practical-rl/modules/00-setup.html](https://project-delphi.github.io/practical-rl/modules/00-setup.html#in-the-lab)

**Next:** [Module 1 · MDPs, returns and the Bellman equation](https://project-delphi.github.io/practical-rl/modules/01-mdps-bellman.html)

In [ ]:
#@title Setup: run this cell first { display-mode: "form" }
# GENERATED by scripts/build_notebooks.py. On Colab it installs exactly what this lab adds
# (keeping every preinstalled package); elsewhere it checks versions and warns.
import importlib.metadata as _md
import os as _os
import subprocess as _sp
import sys as _sys

PRL_REF = "main"  #@param {type:"string"}
WORKED = False  #@param {type:"boolean"}
SEED = 0  #@param {type:"integer"}

_PINS = {"gymnasium": "1.4.0"}
_ALLOW = ["gymnasium"]
_CORE = ["torch", "numpy", "scipy", "pandas", "matplotlib", "transformers", "accelerate", "datasets"]
_PRL_SPEC = _os.environ.get("PRL_SPEC") or "git+https://github.com/project-delphi/practical-rl@" + PRL_REF + "#subdirectory=prl"
if WORKED:
    _os.environ["PRL_WORKED"] = "1"
_on_colab = ("google.colab" in _sys.modules or bool(_os.environ.get("COLAB_RELEASE_TAG"))
             or _os.environ.get("PRL_PLATFORM") == "colab-sim")


def _norm(name):
    return name.lower().replace("_", "-").replace(".", "-")


def _installed_pins(names=None):
    """One name==version per package: the copy Python imports (first on sys.path).

    Colab's sys.path also holds older system copies of some packages; pinning
    every copy would ask pip for two versions of the same package.
    """
    pins = {}
    for d in _md.distributions():
        name = d.metadata["Name"]
        key = _norm(name) if name else None
        if not key or key in pins or key in _keep or (names is not None and key not in names):
            continue
        try:
            pins[key] = f"{name}=={_md.version(name)}"
        except _md.PackageNotFoundError:
            continue
    return sorted(pins.values())


def _pip_install(constraints):
    with open("/tmp/prl-constraints.txt", "w") as fh:
        fh.write("\n".join(constraints) + "\n")
    cmd = [_sys.executable, "-m", "pip", "install", "-q", "-c", "/tmp/prl-constraints.txt",
           *[f"{k}=={val}" for k, val in _PINS.items()], _PRL_SPEC]
    return _sp.run(cmd, capture_output=True, text=True)


if _on_colab:
    _keep = set(_ALLOW) | {"prl"}
    _before = {k: _md.version(k) for k in ("numpy", "torch", "scipy") if k in _sys.modules}
    _r = _pip_install(_installed_pins())
    if _r.returncode != 0:
        # Fall back to protecting only the heavy preinstalled packages.
        print("Keeping every preinstalled package was not possible; keeping the core ones "
              f"({', '.join(_CORE)}) and retrying.")
        print(_r.stderr.strip().splitlines()[-1] if _r.stderr.strip() else "")
        _r = _pip_install(_installed_pins(set(_CORE)))
    if _r.returncode != 0:
        print(_r.stdout[-3000:], _r.stderr[-3000:])
        raise SystemExit("Setup failed: the runtime's packages changed. Tell the instructor.")
    if any(_md.version(k) != val for k, val in _before.items()):
        print("A package that was already imported changed version: Runtime → Restart session, "
              "then run this cell again.")
else:
    for _name, _want in _PINS.items():
        try:
            _have = _md.version(_name)
        except _md.PackageNotFoundError:
            _have = None
        if _have != _want:
            print(f"Warning: this lab expects {_name} {_want}, found {_have}. "
                  "Run `uv sync --group notebooks` in the repository.")

import prl
import prl.plot
import prl.runtime

prl.runtime.print_versions(extra=list(_PINS))
prl.plot.style()
print(f"SEED = {SEED}")

In [ ]:
# GENERATED by scripts/build_notebooks.py. Starts the lab harness; rerunning it is safe.
import prl.lab

lab = prl.lab.init("00-setup", "a99409ccad2a97d5", designed="colab-cpu", api=1, ns=globals(),
                   checkpoints=["0", "1", "2", "3"], packages=["gymnasium"], seat=SEED, badge_ref=PRL_REF)

## Exercise 0 · Runtime check (5 min)

First, find out what you are running on. Every lab this week prints this at the top, and
every run record stores it, so a result always says where it came from.

**Predict.** Before you run the next cell, write down: does this runtime have a GPU? How many CPU cores
do you expect it to have?

In [ ]:
import prl.runtime

rt = prl.runtime.detect()
print(rt.summary())
print("CPU model:", rt.cpu)

**Explain.** Compare with your prediction. Almost every lab this week is designed for a **CPU** runtime:
the networks are small, so speed is limited by stepping the environment, not by matrix
arithmetic. Only the Module 12 stretch and one capstone option are designed for a T4 GPU.

In [ ]:
from prl.checks import m00

lab.check(0, m00.check_runtime, rt)

### How fast is this runtime?

The next cell steps CartPole with random actions, then (if PyTorch is installed) trains a
small network, for a few seconds each. The numbers go into your run record; we use records
like yours to size the lab budgets. You do not need to do anything with them.

In [ ]:
import time

import gymnasium as gym

budget = lab.use_settings(
    prl.runtime.settings({"bench_seconds": 5.0}, {"bench_seconds": 1.0}, designed="colab-cpu")
)

bench_env = gym.make("CartPole-v1")
bench_env.reset(seed=0)
steps, start = 0, time.perf_counter()
while time.perf_counter() - start < budget.bench_seconds:
    _, _, terminated, truncated, _ = bench_env.step(bench_env.action_space.sample())
    steps += 1
    if terminated or truncated:
        bench_env.reset()
env_rate = steps / (time.perf_counter() - start)
lab.metric("cartpole_env_steps_per_s", round(env_rate))
print(f"CartPole: {env_rate:,.0f} environment steps per second")

try:
    import torch

    net = torch.nn.Sequential(
        torch.nn.Linear(4, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, 64),
        torch.nn.ReLU(),
        torch.nn.Linear(64, 2),
    )
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    x, y = torch.randn(64, 4), torch.randn(64, 2)
    updates, start = 0, time.perf_counter()
    while time.perf_counter() - start < budget.bench_seconds:
        loss = torch.nn.functional.mse_loss(net(x), y)
        opt.zero_grad()
        loss.backward()
        opt.step()
        updates += 1
    mlp_rate = updates / (time.perf_counter() - start)
    lab.metric("mlp_updates_per_s", round(mlp_rate))
    lab.metric("torch_threads", torch.get_num_threads())
    print(f"Small MLP: {mlp_rate:,.0f} gradient updates per second (batch 64)")
except ImportError:
    print("PyTorch is not installed here; skipping the network benchmark.")

## Exercise 1 · A random agent (5 min)

A random agent picks every action uniformly at random. It is the floor every method this
week must beat, and checkpoints often ask for "at least 3x random".

**Predict.** CartPole-v1 pays +1 for every step the pole stays up, and stops at 500 steps. What average
return do you expect from a random agent? Write a number.

In [ ]:
def run_episode(env, seed):
    """Play one episode with uniformly random actions and return the total reward."""
    # TODO 1: reset the environment with `seed`, seed its action space too, then step with
    # env.action_space.sample() until the episode is terminated or truncated.
    # Add up the rewards and return the total.
    raise NotImplementedError("TODO 1: write run_episode")

<details><summary>Hint 1</summary>

`obs, info = env.reset(seed=seed)` starts an episode. `env.step(action)` returns
`obs, reward, terminated, truncated, info`. The episode is over when either flag is true.

</details>

In [ ]:
#@title Solution 1 · run_episode (open after trying) { display-mode: "form" }
@lab.solution(1)
def run_episode(env, seed):
    """Play one episode with uniformly random actions and return the total reward."""
    env.reset(seed=seed)
    env.action_space.seed(seed)
    total, done = 0.0, False
    while not done:
        _, reward, terminated, truncated, _ = env.step(env.action_space.sample())
        total += float(reward)
        done = terminated or truncated
    return total

In [ ]:
import numpy as np

env = gym.make("CartPole-v1")
returns = [run_episode(env, seed=SEED * 1000 + i) for i in range(20)]
print(
    f"Mean return over 20 episodes: {np.mean(returns):.1f} "
    f"(min {min(returns):.0f}, max {max(returns):.0f})"
)

**Explain.** How close was your guess? Why does random play end so quickly, and why do some episodes
last much longer than others?

In [ ]:
lab.check(1, m00.check_episode_returns, returns, 20)

## Exercise 2 · Seed bands (5 min)

One run is one draw from a random process. To say anything about a method you need several
seeds, and you show their spread.

**Predict.** You will run 5 seeds of 20 episodes each. Will the five seeds' mean returns agree to within
1 point? Within 5? Write your guess.

In [ ]:
def returns_by_seed(n_seeds, n_episodes, base_seed):
    """Return an array of shape (n_seeds, n_episodes) of random-agent returns.

    Seed s uses episode seeds base_seed + 100 * s + i for i in range(n_episodes).
    """
    # TODO 2: call run_episode for each seed and episode; return a NumPy array.
    raise NotImplementedError("TODO 2: write returns_by_seed")

<details><summary>Hint 2</summary>

A nested list comprehension works: `[[run_episode(env, ...) for i in ...] for s in ...]`,
then `np.array(...)`.

</details>

In [ ]:
#@title Solution 2 · returns_by_seed (open after trying) { display-mode: "form" }
@lab.solution(2)
def returns_by_seed(n_seeds, n_episodes, base_seed):
    """Return an array of shape (n_seeds, n_episodes) of random-agent returns."""
    return np.array(
        [
            [run_episode(env, base_seed + 100 * s + i) for i in range(n_episodes)]
            for s in range(n_seeds)
        ]
    )

In [ ]:
import matplotlib.pyplot as plt

by_seed = returns_by_seed(5, 20, SEED * 1000)
running_mean = np.cumsum(by_seed, axis=1) / np.arange(1, 21)
prl.plot.curves(
    np.arange(1, 21),
    {"random agent": running_mean},
    xlabel="episodes",
    ylabel="running mean return",
    title="Random agent on CartPole-v1, 5 seeds",
    budget="5 seeds x 20 episodes",
)
plt.show()
seed_means = by_seed.mean(axis=1)
print("Per-seed mean returns:", np.round(seed_means, 1))
lab.seeds([SEED * 1000 + 100 * s for s in range(5)])
lab.metric("random_mean_return", round(float(by_seed.mean()), 2))

**Explain.** How far apart are the per-seed means? Suppose someone reported "my agent scores 25, the
random agent scores 22" from one seed each. What would you ask them?

In [ ]:
lab.check(2, m00.check_seed_bands, by_seed, 5, 20)

## Exercise 3 · Your run record (5 min)

Each lab ends by printing a **run record**: what ran, where, with which versions, and which
checkpoints passed. The site's readiness page is built only from records like this one.

**Predict.** Which fields of the record would tell a reader whether this ran on Colab, and on a GPU?

In [ ]:
rec = lab.build_record()
print(
    "platform:", rec["platform"], "| env:", rec["env"], "| accelerator:", rec["hardware"]["accel"]
)
print("checkpoints so far:", [(c["label"], c["pass"]) for c in rec["checkpoints"]])

**Explain.** The record never includes your name, paths or keys. On Colab the last cell offers it as a
download; send it to your instructor if they ask for setup evidence.

In [ ]:
lab.check(3, m00.check_record, rec)

In [ ]:
# GENERATED by scripts/build_notebooks.py. Prints your summary and your run record.
lab.finish()